# Challenge 4 · Teleport the Qubit
**Tier 2 - Quantum Algorithms and Applications** · **Estimated time: 40-60 minutes** (an estimate, not a guarantee) · *Participant notebook*

> Alice must send an unknown qubit state to Bob. She may not look at it, copy it, or fly it over - but she can share an entangled pair in advance and send two ordinary bits by phone.

### Learning objectives
- Implement the three stages of teleportation: entanglement, Bell-basis measurement, correction.
- Explain why two classical bits are necessary and why nothing travels faster than light.
- Use classically controlled gates in Qiskit.
- Verify a protocol on many input states, not on a single lucky one.

### Prerequisites
- Challenges 1-3 or equivalent
- Bell states
- Mid-circuit measurement and classical control (`if_test`) are introduced in the notebook

### What you will write
| Function | Status |
|---|---|
| `create_bell_pair` | required |
| `alice_bell_measurement` | required |
| `bob_correction` | required |
| `average_bob_state_without_correction` | bonus (optional) |

### Acceptance criteria
Your solution is accepted when all **core** and **edge-case** requirements hold:
- `create_bell_pair` leaves the message qubit untouched and puts the two named qubits in |Phi+>.
- `alice_bell_measurement` performs CNOT, H and both measurements; all four outcomes are equally likely and Bob's state in branch (z, x) is X^x Z^z |psi>.
- `bob_correction` restores |psi> for each of the four classical outcomes using classically controlled gates.
- The assembled protocol teleports many different states with fidelity >= 1 - 1e-9 in every branch.
- The functions work for any distinct qubit indices (nothing hard-coded) and reject invalid ones.

**Bonus (optional, never required to pass):**
- Without the classical bits Bob's average state is maximally mixed for every input (no signalling).

**How this notebook works.** Run the cells from top to bottom. Cells marked **TODO** are yours. After each part run the **public checks**: they tell you what works and what does not. The instructors run additional hidden tests, so also try your own edge cases. Hints go from subtle to explicit - open them one at a time.

In [ ]:
# Setup - safe to re-run (works in Google Colab and in a local Jupyter session)
import importlib, importlib.util, pathlib, subprocess, sys

if importlib.util.find_spec('qiskit_fest') is None:
    here = pathlib.Path.cwd().resolve()
    # the project is the working folder, one of its parents, or (Colab: working folder /content) a folder right below it
    candidates = [here, *here.parents, *sorted(p for p in here.iterdir() if p.is_dir())]
    root = next((p for p in candidates
                 if (p / 'pyproject.toml').is_file() and (p / 'src' / 'qiskit_fest').is_dir()), None)
    if root is None:
        raise RuntimeError("Open this notebook from inside the qiskit-fall-fest-challenges folder "
                           "(see README.md, section 'Colab').")
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(root)])
    except subprocess.CalledProcessError:
        print('pip could not install the project (no internet?) - continuing with the libraries that are already installed.')
    # a running kernel does not re-read the .pth file that pip has just written, so make the package importable here and now
    sys.path.insert(0, str(root / 'src'))
    importlib.invalidate_caches()

from qiskit_fest.checks import run_public_checks, if_ready
from qiskit_fest.env import print_environment
print_environment()

## The problem
**Alice** holds a qubit in an unknown state |ψ⟩. **Bob** is far away. They share an entangled pair prepared in advance, and Alice may send **two ordinary classical bits** (a phone call).
Complete the three stages of the *teleportation protocol* so that Bob's qubit ends up in exactly the state |ψ⟩:

1. **Entanglement** - Alice and Bob share a Bell pair (`create_bell_pair`).
2. **Bell-basis measurement** - Alice entangles her message qubit with her half of the pair, rotates, and measures both qubits (`alice_bell_measurement`). Two classical bits result.
3. **Correction** - Bob applies Pauli gates *controlled by the classical bits* (`bob_correction`).

**Roles in the 3-qubit circuit:** qubit 0 = message, qubit 1 = Alice's half, qubit 2 = Bob. The two classical bits are named `mz` (outcome of measuring the *message* qubit; controls Z) and `mx` (outcome of measuring *Alice's half*; controls X).

### Function contracts
All three functions **modify the given circuit `qc` in place** (their return value is ignored) and must use the qubit indices they are given - never hard-coded numbers.

| Function | What it must do | Invalid input |
|---|---|---|
| `create_bell_pair(qc, a, b)` | gates only: turn qubits `a`, `b` (starting in &#124;00⟩) into &#124;Φ+⟩ = (&#124;00⟩+&#124;11⟩)/√2; all other qubits untouched | equal / negative / out-of-range indices raise `ValueError` or Qiskit's `CircuitError` |
| `alice_bell_measurement(qc, msg, alice, z_bit, x_bit)` | `CX(msg → alice)`, `H(msg)`, then measure `msg` into `z_bit` and `alice` into `x_bit` (`z_bit`, `x_bit` are `Clbit` objects) | equal / out-of-range qubit indices raise `ValueError` or `CircuitError` |
| `bob_correction(qc, bob, z_bit, x_bit)` | classically controlled gates: apply X if `x_bit` is 1, then Z if `z_bit` is 1 (use `qc.if_test`) | out-of-range `bob` raises `ValueError` or `CircuitError` |
| *(bonus)* `average_bob_state_without_correction(prep)` | `DensityMatrix` of Bob's qubit **averaged over Alice's outcomes**, if Bob never hears from Alice | - |

**Tolerances.** The whole protocol is simulated *exactly* (no sampling): Bob's state in every branch must match |ψ⟩ with fidelity ≥ 1 − 1e-9.

## Concept: what the protocol does
For a message $|\psi\rangle=\alpha|0\rangle+\beta|1\rangle$ and the shared pair $|\Phi^+\rangle$, applying CNOT (message → Alice) and H (message) rewrites the joint state as
$$\tfrac12\sum_{z,x\in\{0,1\}}|z\rangle_{\rm msg}|x\rangle_{\rm Alice}\;\otimes\; X^{x}Z^{z}|\psi\rangle_{\rm Bob}.$$
* Alice's measurement returns one of four outcomes $(z,x)$, **each with probability ¼**, and Bob's qubit is then $X^xZ^z|\psi\rangle$ - the message *scrambled by a known Pauli*.
* **Two bits are needed** because there are four possibilities. Bob undoes the scrambling: first $X^x$, then $Z^z$.
* **Nothing travels faster than light.** Before the bits arrive, Bob's qubit is maximally mixed (I/2) *whatever* |ψ⟩ was - he can learn nothing. And Alice's original qubit is destroyed by her measurement: the state is **moved**, not copied (no-cloning).

**Classical control in Qiskit.** `qc.measure(qubit, clbit)` stores a measurement result in a classical bit. A block such as
```python
with qc.if_test((clbit, 1)):
    qc.x(target)          # runs only if the classical bit is 1
```
applies gates *conditioned on the result*. The next cell shows it on a tiny example, simulated **exactly** with the helper `exact_branches` (one "branch" per measurement outcome).

In [ ]:
import numpy as np
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit.quantum_info import DensityMatrix, Statevector
from qiskit_fest.challenges.c04_teleportation import assemble
from qiskit_fest.sim import average_reduced_state, exact_branches

In [ ]:
from qiskit_fest.challenges.c04_teleportation import EXAMPLE_STATES, branch_report, plot_branch_fidelities, prep_circuit
from qiskit_fest.sim import measured_distribution
from qiskit_fest.viz import draw_circuit

# A tiny classically-controlled circuit: measure qubit 0; if the result is 1, flip qubit 1; then measure qubit 1.
demo = QuantumCircuit(QuantumRegister(2, "q"), ClassicalRegister(1, "c"), ClassicalRegister(1, "out"))
demo.h(0)
demo.measure(0, demo.clbits[0])
with demo.if_test((demo.clbits[0], 1)):
    demo.x(1)
demo.measure(1, demo.clbits[1])
draw_circuit(demo, "Classical control: qubit 1 copies the measurement result of qubit 0")
print("exact outcome probabilities (bit-string 'out c'):", measured_distribution(demo))

## Part 1 · Share a Bell pair  *(TODO)*
Complete `create_bell_pair(qc, a, b)`. It must add gates to `qc` that turn qubits `a` and `b` into |Φ+⟩ and leave every other qubit alone.

<details><summary>💡 Hint 1 (subtle)</summary>

Which two gates turn |00⟩ into (|00⟩ + |11⟩)/√2? One makes a superposition; the other makes the second qubit follow the first.
</details>

<details><summary>💡 Hint 2</summary>

Use the indices you are given: `a` and `b` are *parameters*, not the numbers 1 and 2. The tests call your function with many different pairs.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Hadamard on `a`, then a CNOT with control `a` and target `b`. Add a check that `a != b` and that both are valid indices of `qc` (raise `ValueError`).
</details>

In [ ]:
def create_bell_pair(qc: QuantumCircuit, a: int, b: int) -> None:
    """Add gates to `qc` that turn qubits a, b (in |00>) into (|00>+|11>)/sqrt(2). Other qubits untouched."""
    # TODO: validate the indices (ValueError), then add the two gates.
    raise NotImplementedError("TODO: implement create_bell_pair()")

In [ ]:
report = run_public_checks(4, globals(), keyword="create_bell_pair")

## Part 2 · Alice's Bell-basis measurement  *(TODO)*
Complete `alice_bell_measurement(qc, msg, alice, z_bit, x_bit)`:
1. `CX` with control `msg`, target `alice`;
2. `H` on `msg`;
3. measure `msg` into `z_bit` and `alice` into `x_bit`.

After this, the four classical outcomes are equally likely and Bob's qubit is $X^{x}Z^{z}|\psi\rangle$. **The assignment of the two bits matters** - Bob's correction (Part 3) relies on it.

<details><summary>💡 Hint 1 (subtle)</summary>

Alice's two qubits are `msg` and `alice`. She *entangles* them, rotates one, and then reads both out. Think about which of the two gates acts first.
</details>

<details><summary>💡 Hint 2</summary>

`qc.measure(qubit, clbit)` stores the outcome of `qubit` in the classical bit `clbit`. `z_bit` is for the **message** qubit, `x_bit` for **Alice's half**.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Order: `cx(msg, alice)`, `h(msg)`, `measure(msg, z_bit)`, `measure(alice, x_bit)`. Validate that `msg` and `alice` are distinct valid indices.
</details>

In [ ]:
def alice_bell_measurement(qc: QuantumCircuit, msg: int, alice: int, z_bit, x_bit) -> None:
    """CX(msg -> alice), H(msg), then measure msg into z_bit and alice into x_bit."""
    # TODO: validate the indices (ValueError), then add the gates and the two measurements.
    raise NotImplementedError("TODO: implement alice_bell_measurement()")

In [ ]:
report = run_public_checks(4, globals(), keyword="alice_bell_measurement")

## Part 3 · Bob's correction  *(TODO)*
Complete `bob_correction(qc, bob, z_bit, x_bit)`. Bob holds $X^{x}Z^{z}|\psi\rangle$. Undo it with **classically controlled** gates: apply `X` when `x_bit` is 1, then `Z` when `z_bit` is 1.

<details><summary>💡 Hint 1 (subtle)</summary>

Bob's qubit is the message after Pauli gates chosen by Alice's measurement results. Every Pauli gate is its own inverse.
</details>

<details><summary>💡 Hint 2</summary>

`with qc.if_test((bit, 1)):` followed by an indented gate call applies the gate only when the classical bit equals 1.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Two `if_test` blocks: first on `x_bit` with an `x` gate on `bob`, then on `z_bit` with a `z` gate on `bob`.
</details>

In [ ]:
def bob_correction(qc: QuantumCircuit, bob: int, z_bit, x_bit) -> None:
    """Apply X if x_bit == 1, then Z if z_bit == 1, to qubit `bob` (classically controlled)."""
    # TODO: validate `bob` (ValueError), then add the two classically controlled gates.
    raise NotImplementedError("TODO: implement bob_correction()")

In [ ]:
report = run_public_checks(4, globals(), keyword="bob_correction")

## Put it together and verify  *(provided)*
`build_teleportation_circuit(prep)` wires your three functions into the full protocol. The check below simulates **every** measurement branch exactly and reports the fidelity of Bob's qubit with the message.
A correct protocol gives **fidelity 1 in all four branches, for every message** - the test states include ones where a *missing* correction would not be noticed on a lucky example (for instance |+⟩ is unchanged by X).

In [ ]:
def build_teleportation_circuit(prep: QuantumCircuit) -> QuantumCircuit:
    """The complete protocol built from the three functions above (qubits: 0 message, 1 Alice, 2 Bob)."""
    return assemble(prep, create_bell_pair, alice_bell_measurement, bob_correction)

In [ ]:
with if_ready():
    circuit = build_teleportation_circuit(prep_circuit(EXAMPLE_STATES["generic"]))
    draw_circuit(circuit, "Teleportation circuit (dynamic: classically controlled corrections)")
    for name, psi in EXAMPLE_STATES.items():
        rows = branch_report(build_teleportation_circuit(prep_circuit(psi)), psi)
        print(f"message {name:8s} -> branch fidelities:", [round(f, 6) for _, _, f in rows], " probabilities:", [round(p, 3) for _, p, _ in rows])
    plot_branch_fidelities(circuit, EXAMPLE_STATES["generic"])

In [ ]:
report = run_public_checks(4, globals(), keyword="protocol")

## Bonus · No signalling  *(optional)*
Complete `average_bob_state_without_correction(prep)`: build the protocol **without** Bob's correction (Parts 1-2 only), simulate every branch, and return Bob's **averaged** reduced state $\sum_{\rm outcomes} p\,\rho_{\rm Bob}$ as a `DensityMatrix`.
It should be the maximally mixed state I/2 **for every message** - Bob has learned nothing until the classical bits arrive.

<details><summary>💡 Hint</summary>

Build the circuit with `QuantumRegister(3)` and two 1-bit `ClassicalRegister`s, call your two functions, run `exact_branches`, and average with `average_reduced_state(branches, [2])`.
</details>

In [ ]:
def average_bob_state_without_correction(prep: QuantumCircuit) -> DensityMatrix:
    """Bob's qubit averaged over Alice's measurement outcomes when no correction is applied (should be I/2)."""
    # TODO (bonus): same as the protocol, but stop before Bob's correction and average over the branches.
    raise NotImplementedError("TODO (bonus): implement average_bob_state_without_correction()")

In [ ]:
with if_ready():
    for name, psi in EXAMPLE_STATES.items():
        rho = average_bob_state_without_correction(prep_circuit(psi))
        print(f"message {name:8s}: Bob's averaged state = {np.round(rho.data, 6).tolist()}")

In [ ]:
report = run_public_checks(4, globals(), keyword="bonus")

## Reflection
1. Why does Alice need to send **two** classical bits? What would happen if she sent only one?
2. Teleportation uses entanglement, but entanglement alone does not let Alice send a message. Which step of your bonus experiment shows that?
3. After the protocol Alice's qubit holds a definite classical value. Why is this consistent with the no-cloning theorem?
4. What exactly was "teleported": the particle, the information, or the state? Which resources were consumed?

### Hand-in
When you are done, save your functions as a file (or simply hand in this notebook).

In [ ]:
# Optional: save your functions as a stand-alone file you can hand in
from qiskit_fest.export import export_submission
export_submission(globals(), 4)